# Group-Level SSVEP Data Quality Analysis

Group-level summary of data quality across all subjects, building on the single-subject pipeline in `analyze_ssvep.ipynb`.

In [ ]:
import os
import numpy as np
import pandas as pd
import mne
from mne_bids import BIDSPath, read_raw_bids
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportion_confint
from sklearn.cross_decomposition import CCA
from scipy.linalg import eigh
from scipy.signal import cheby1, detrend, sosfiltfilt

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['font.family'] = 'Times New Roman'

BIDS_ROOT = 'dataset/derivatives/preprocessed'
TASK = 'ssvep'
DATATYPE = 'eeg'
SUFFIX = 'eeg'

TARGET_FREQS = [10.0, 12.0, 15.0]
ANALYSIS_TMIN = 0.5
ANALYSIS_TMAX = 4.5
MAX_ANALYSIS_FREQ = 50.0
SNR_NOISE_BW = 1.0
DECISION_WINDOW_S = ANALYSIS_TMAX - ANALYSIS_TMIN
FULL_TRIAL_CYCLE_S = 3.0 + 1.0 + 5.0 + 5.0

QC_PTP_UV_THRESHOLD = 150.0
QC_RMS_UV_THRESHOLD = 50.0

OCCIPITAL_CHS = ['O1', 'Oz', 'O2', 'Pz', 'P3', 'P4']

AUX_CHANNEL_TYPES = {
    'REOG': 'eog', 'LEOG': 'eog',
    'EMG1': 'emg', 'EMG2': 'emg', 'EMG3': 'emg', 'EMG4': 'emg', 'EMG5': 'emg',
    'EMG6': 'emg', 'EMG7': 'emg', 'EMG8': 'emg', 'EMG9': 'emg',
    'SpO2': 'misc', 'PulseRate': 'misc', 'Pleth': 'misc',
}

## 1. Utility Functions

In [ ]:
def parse_stim_label(event_name):
    event_name = str(event_name)
    parts = event_name.split('_')
    if len(parts) != 4 or parts[0] != 'stim':
        raise ValueError(f'Not a valid stim label: {event_name}')
    return {
        'Event': event_name,
        'Frequency': float(parts[1].replace('Hz', '')),
        'Transparency': int(parts[2].replace('trans', '')),
        'Color': parts[3],
    }


def valid_harmonics(target_freq, max_freq=MAX_ANALYSIS_FREQ, max_harmonics=3):
    return [target_freq * h for h in range(1, max_harmonics + 1) if target_freq * h <= max_freq + 1e-9]


def calculate_harmonic_snr(psd_data, freqs, target_freq, noise_bw=SNR_NOISE_BW,
                           max_freq=MAX_ANALYSIS_FREQ, max_harmonics=3):
    psd_array = np.asarray(psd_data)
    was_1d = psd_array.ndim == 1
    psd_2d = np.atleast_2d(psd_array)
    freqs = np.asarray(freqs)
    bin_width = np.nanmedian(np.diff(freqs))
    guard_hz = max(0.25, 1.5 * bin_width)

    harmonic_ratios = []
    for harmonic_freq in valid_harmonics(target_freq, max_freq=max_freq, max_harmonics=max_harmonics):
        idx_target = int(np.argmin(np.abs(freqs - harmonic_freq)))
        noise_mask = (np.abs(freqs - harmonic_freq) <= noise_bw) & (np.abs(freqs - harmonic_freq) > guard_hz)
        if not np.any(noise_mask):
            continue
        target_pow = psd_2d[:, idx_target]
        noise_pow = np.nanmean(psd_2d[:, noise_mask], axis=1)
        harmonic_ratios.append(target_pow / noise_pow)

    if not harmonic_ratios:
        snr_linear = np.full(psd_2d.shape[0], np.nan)
    else:
        snr_linear = np.nanmean(np.vstack(harmonic_ratios), axis=0)
    snr_db = 10 * np.log10(snr_linear)

    if was_1d:
        return float(snr_linear[0]), float(snr_db[0])
    return snr_linear, snr_db


def calculate_fft_itpc(data, sfreq, target_freq):
    data = np.asarray(data)
    if data.shape[0] < 2:
        return np.nan
    fft_vals = np.fft.rfft(data, axis=-1)
    fft_freqs = np.fft.rfftfreq(data.shape[-1], d=1.0 / sfreq)
    idx = int(np.argmin(np.abs(fft_freqs - target_freq)))
    phase = np.angle(fft_vals[:, :, idx])
    itpc_ch = np.abs(np.mean(np.exp(1j * phase), axis=0))
    return float(np.nanmean(itpc_ch))


def discover_subjects(bids_root):
    subjects = []
    for entry in sorted(os.listdir(bids_root)):
        if entry.startswith('sub-') and os.path.isdir(os.path.join(bids_root, entry)):
            eeg_dir = os.path.join(bids_root, entry, 'eeg')
            if os.path.isdir(eeg_dir):
                subjects.append(entry.replace('sub-', ''))
    return subjects


def get_reference_signals(freq, sfreq, n_samples, max_freq=MAX_ANALYSIS_FREQ, max_harmonics=3):
    t = np.arange(n_samples) / sfreq
    ref = []
    for harmonic_freq in valid_harmonics(freq, max_freq=max_freq, max_harmonics=max_harmonics):
        ref.append(np.sin(2 * np.pi * harmonic_freq * t))
        ref.append(np.cos(2 * np.pi * harmonic_freq * t))
    return np.array(ref).T


def prepare_cca_data(data):
    x = detrend(data, axis=1).T
    x = x - np.mean(x, axis=0, keepdims=True)
    scale = np.std(x, axis=0, keepdims=True)
    scale[scale == 0] = 1.0
    return x / scale


def cca_analysis(data, freqs, sfreq, max_freq=MAX_ANALYSIS_FREQ, max_harmonics=3):
    n_samples = data.shape[1]
    x = prepare_cca_data(data)
    scores = {}
    for freq in freqs:
        ref = get_reference_signals(freq, sfreq, n_samples, max_freq=max_freq, max_harmonics=max_harmonics)
        cca = CCA(n_components=1, max_iter=1000)
        cca.fit(x, ref)
        x_score, y_score = cca.transform(x, ref)
        corr = np.corrcoef(x_score[:, 0], y_score[:, 0])[0, 1]
        scores[freq] = corr if np.isfinite(corr) else -np.inf
    return scores


FBCCA_LOW_CUTS = [6.0, 14.0, 22.0, 30.0]


def filter_bank(data, sfreq, band_idx, max_freq=MAX_ANALYSIS_FREQ):
    nyq = sfreq / 2
    low = FBCCA_LOW_CUTS[band_idx]
    high = min(max_freq, nyq - 1.0)
    if low >= high:
        raise ValueError(f'Invalid sub-band: low={low}, high={high}')
    sos = cheby1(4, 1, [low / nyq, high / nyq], btype='bandpass', output='sos')
    return sosfiltfilt(sos, data, axis=-1)


def fbcca_analysis(data, freqs, sfreq, n_bands=4):
    n_bands = min(n_bands, len(FBCCA_LOW_CUTS))
    n_samples = data.shape[1]
    rho = np.zeros((n_bands, len(freqs)))
    weights = np.array([np.power(i + 1, -1.25) + 0.25 for i in range(n_bands)])
    for band_idx in range(n_bands):
        filtered_data = filter_bank(data, sfreq, band_idx)
        x = prepare_cca_data(filtered_data)
        for freq_idx, freq in enumerate(freqs):
            ref = get_reference_signals(freq, sfreq, n_samples)
            cca = CCA(n_components=1, max_iter=1000)
            cca.fit(x, ref)
            x_score, y_score = cca.transform(x, ref)
            corr = np.corrcoef(x_score[:, 0], y_score[:, 0])[0, 1]
            rho[band_idx, freq_idx] = corr if np.isfinite(corr) else 0.0
    final_scores = np.dot(weights, np.square(rho))
    return {freqs[i]: final_scores[i] for i in range(len(freqs))}


def demean_trials(data):
    return data - data.mean(axis=-1, keepdims=True)


def train_trca_filters(train_data, train_labels, target_freqs):
    filters = {}
    templates = {}
    n_channels = train_data.shape[1]
    for freq in target_freqs:
        x_class = demean_trials(train_data[train_labels == freq])
        if len(x_class) < 2:
            continue
        s_mat = np.zeros((n_channels, n_channels))
        q_mat = np.zeros((n_channels, n_channels))
        for i in range(len(x_class)):
            xi = x_class[i]
            q_mat += xi @ xi.T
            for j in range(i + 1, len(x_class)):
                xj = x_class[j]
                s_mat += xi @ xj.T + xj @ xi.T
        reg = 1e-6 * np.trace(q_mat) / n_channels if np.trace(q_mat) > 0 else 1e-6
        eigvals, eigvecs = eigh(s_mat, q_mat + reg * np.eye(n_channels))
        w = eigvecs[:, np.argmax(eigvals)]
        filters[freq] = w / (np.linalg.norm(w) + 1e-12)
        templates[freq] = x_class.mean(axis=0)
    return filters, templates


def trca_classify(data, filters, templates):
    x = demean_trials(np.asarray(data))
    scores = {}
    for freq, w in filters.items():
        test_proj = w @ x
        template_proj = w @ templates[freq]
        corr = np.corrcoef(test_proj, template_proj)[0, 1]
        scores[freq] = corr if np.isfinite(corr) else -np.inf
    return scores


def train_fbtrca_filters(train_data, train_labels, target_freqs, sfreq, n_bands=4):
    n_bands = min(n_bands, len(FBCCA_LOW_CUTS))
    n_channels = train_data.shape[1]
    all_filters = {freq: [] for freq in target_freqs}
    all_templates = {freq: [] for freq in target_freqs}

    for band_idx in range(n_bands):
        filtered_data = np.stack([
            filter_bank(train_data[k], sfreq, band_idx) for k in range(len(train_data))
        ])
        for freq in target_freqs:
            x_class = demean_trials(filtered_data[train_labels == freq])
            if len(x_class) < 2:
                all_filters[freq].append(np.zeros(n_channels))
                all_templates[freq].append(np.zeros_like(x_class[0]))
                continue
            s_mat = np.zeros((n_channels, n_channels))
            q_mat = np.zeros((n_channels, n_channels))
            for i in range(len(x_class)):
                xi = x_class[i]
                q_mat += xi @ xi.T
                for j in range(i + 1, len(x_class)):
                    xj = x_class[j]
                    s_mat += xi @ xj.T + xj @ xi.T
            reg = 1e-6 * np.trace(q_mat) / n_channels if np.trace(q_mat) > 0 else 1e-6
            eigvals, eigvecs = eigh(s_mat, q_mat + reg * np.eye(n_channels))
            w = eigvecs[:, np.argmax(eigvals)]
            w = w / (np.linalg.norm(w) + 1e-12)
            all_filters[freq].append(w)
            all_templates[freq].append(x_class.mean(axis=0))

    return all_filters, all_templates


def fbtrca_classify(data, all_filters, all_templates, target_freqs, sfreq, n_bands=4):
    n_bands = min(n_bands, len(FBCCA_LOW_CUTS))
    weights = np.array([np.power(i + 1, -1.25) + 0.25 for i in range(n_bands)])
    final_scores = {}

    for freq in target_freqs:
        weighted_corr = 0.0
        for band_idx in range(n_bands):
            filtered = filter_bank(data, sfreq, band_idx)
            x = demean_trials(filtered)
            w = all_filters[freq][band_idx]
            template = all_templates[freq][band_idx]
            test_proj = w @ x
            template_proj = w @ template
            corr = np.corrcoef(test_proj, template_proj)[0, 1]
            if np.isfinite(corr):
                weighted_corr += weights[band_idx] * corr ** 2
        final_scores[freq] = weighted_corr
    return final_scores


def score_margin(scores):
    ordered = sorted(scores.values(), reverse=True)
    if len(ordered) < 2:
        return np.nan
    return float(ordered[0] - ordered[1])


def calculate_itr(n_targets, accuracy, decision_seconds):
    accuracy = float(np.clip(accuracy, 1e-12, 1 - 1e-12))
    if accuracy <= 1 / n_targets:
        return 0.0
    bits = (
            np.log2(n_targets)
            + accuracy * np.log2(accuracy)
            + (1 - accuracy) * np.log2((1 - accuracy) / (n_targets - 1))
    )
    return float(bits * 60 / decision_seconds)


def summarize_accuracy(df, group_cols=None, acc_col='Correct'):
    if group_cols is None:
        group_cols = []
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    if group_cols:
        grouped = df.groupby(group_cols)[acc_col].agg(Correct='sum', Trials='count', Accuracy='mean').reset_index()
    else:
        grouped = pd.DataFrame({
            'Correct': [df[acc_col].sum()],
            'Trials': [df[acc_col].count()],
            'Accuracy': [df[acc_col].mean()],
        })
    ci = grouped.apply(
        lambda row: proportion_confint(int(row['Correct']), int(row['Trials']), alpha=0.05, method='wilson'),
        axis=1,
    )
    grouped['CI95_Low'] = [x[0] for x in ci]
    grouped['CI95_High'] = [x[1] for x in ci]
    return grouped

## 2. Discover Subjects

In [ ]:
all_subjects = discover_subjects(BIDS_ROOT)
print(f'Found {len(all_subjects)} subjects: {all_subjects}')

## 3. Single-Subject Processing Function

Encapsulate data loading, preprocessing, and quality metric extraction for batch processing.

In [ ]:
def process_subject(subject_id):
    result = {
        'Subject': subject_id,
        'Status': 'pending',
        'Error': None,
        'N_Trials': 0,
        'N_Channels': 0,
        'Sfreq': None,
        'snr_rows': [],
        'snr_harmonic_rows': [],
        'quality_rows': [],
        'itpc_rows': [],
        'classification_rows': [],
        'avg_psd': None,
        'psd_freqs': None,
    }

    try:
        bids_path = BIDSPath(
            subject=subject_id, task=TASK, suffix=SUFFIX,
            datatype=DATATYPE, root=BIDS_ROOT,
        )
        raw = read_raw_bids(bids_path=bids_path, verbose=False)
        raw.load_data()

        result['Sfreq'] = raw.info['sfreq']
        result['N_Channels'] = len(raw.ch_names)

        present_aux_types = {ch: ch_type for ch, ch_type in AUX_CHANNEL_TYPES.items() if ch in raw.ch_names}
        if present_aux_types:
            try:
                raw.set_channel_types(present_aux_types, on_unit_change='ignore')
            except TypeError:
                raw.set_channel_types(present_aux_types)

        all_events, all_event_dict = mne.events_from_annotations(raw, verbose=False)
        event_dict = {str(name): code for name, code in all_event_dict.items()}
        stim_event_id = {name: code for name, code in event_dict.items() if name.startswith('stim_')}

        if not stim_event_id:
            result['Status'] = 'no_stim_events'
            result['Error'] = 'No stim_* events found'
            return result

        epochs_stim_full = mne.Epochs(
            raw, all_events, event_id=stim_event_id,
            tmin=ANALYSIS_TMIN, tmax=ANALYSIS_TMAX,
            baseline=None, preload=True, verbose=False,
        )

        available_chs = [ch for ch in OCCIPITAL_CHS if ch in epochs_stim_full.ch_names]
        if not available_chs:
            result['Status'] = 'no_occipital_chs'
            result['Error'] = f'No occipital channels found. Available: {epochs_stim_full.ch_names}'
            return result

        epochs = epochs_stim_full.copy().pick_channels(available_chs)
        result['N_Trials'] = len(epochs)

        spectrum = epochs.compute_psd(
            method='welch', fmin=5, fmax=MAX_ANALYSIS_FREQ,
            n_fft=1000, n_overlap=500, verbose=False,
        )
        psds, freqs = spectrum.get_data(return_freqs=True)
        result['avg_psd'] = psds.mean(axis=0)
        result['psd_freqs'] = freqs
        id_to_name = {code: name for name, code in epochs.event_id.items()}

        for i, etype in enumerate(epochs.events[:, 2]):
            label = id_to_name[int(etype)]
            info = parse_stim_label(label)
            trial_psds = psds[i]
            snr_linear_ch, snr_db_ch = calculate_harmonic_snr(trial_psds, freqs, info['Frequency'])

            result['snr_rows'].append({
                'Subject': subject_id,
                'Trial_Index': i,
                **info,
                'SNR_linear': float(np.nanmean(snr_linear_ch)),
                'SNR_dB': float(np.nanmean(snr_db_ch)),
            })

            for harmonic_order, harmonic_freq in enumerate(valid_harmonics(info['Frequency']), start=1):
                snr_lin_ch, snr_db_ch_h = calculate_harmonic_snr(
                    trial_psds, freqs, harmonic_freq,
                    max_freq=MAX_ANALYSIS_FREQ, max_harmonics=1,
                )
                result['snr_harmonic_rows'].append({
                    'Subject': subject_id,
                    'Trial_Index': i,
                    **info,
                    'Harmonic_Order': harmonic_order,
                    'Harmonic_Freq': harmonic_freq,
                    'SNR_linear': float(np.nanmean(snr_lin_ch)),
                    'SNR_dB': float(np.nanmean(snr_db_ch_h)),
                })

        epoch_data_uv = epochs.get_data() * 1e6
        for i, etype in enumerate(epochs.events[:, 2]):
            label = id_to_name[int(etype)]
            info = parse_stim_label(label)
            trial = epoch_data_uv[i]
            ch_ptp = np.ptp(trial, axis=1)
            ch_rms = np.sqrt(np.mean(trial ** 2, axis=1))
            result['quality_rows'].append({
                'Subject': subject_id,
                'Trial_Index': i,
                **info,
                'Mean_PTP_uV': float(np.nanmean(ch_ptp)),
                'Max_PTP_uV': float(np.nanmax(ch_ptp)),
                'Mean_RMS_uV': float(np.nanmean(ch_rms)),
                'Max_RMS_uV': float(np.nanmax(ch_rms)),
                'Likely_Artifact': int(
                    (np.nanmax(ch_ptp) > QC_PTP_UV_THRESHOLD) or (np.nanmax(ch_rms) > QC_RMS_UV_THRESHOLD)
                ),
            })

        sfreq = epochs.info['sfreq']
        for label in epochs.event_id.keys():
            info = parse_stim_label(label)
            data = epochs[label].get_data()
            harmonic_itpcs = []
            for harmonic_order, harmonic_freq in enumerate(valid_harmonics(info['Frequency']), start=1):
                itpc_value = calculate_fft_itpc(data, sfreq, harmonic_freq)
                harmonic_itpcs.append(itpc_value)
                result['itpc_rows'].append({
                    'Subject': subject_id,
                    **info,
                    'Trials': data.shape[0],
                    'Harmonic_Order': harmonic_order,
                    'Harmonic_Freq': harmonic_freq,
                    'ITPC': itpc_value,
                })
            result['itpc_rows'].append({
                'Subject': subject_id,
                **info,
                'Trials': data.shape[0],
                'Harmonic_Order': 0,
                'Harmonic_Freq': np.nan,
                'ITPC': float(np.nanmean(harmonic_itpcs)),
            })

        data_all = epochs.get_data()
        labels = []
        meta_list = []
        for event_code in epochs.events[:, 2]:
            info_c = parse_stim_label(id_to_name[int(event_code)])
            labels.append(info_c['Frequency'])
            meta_list.append(info_c)
        labels = np.array(labels, dtype=float)

        for i, info_c in enumerate(meta_list):
            trial_data = data_all[i]
            row_base = {
                'Subject': subject_id,
                'Trial_Index': i,
                **info_c,
                'True_Freq': info_c['Frequency'],
            }

            cca_scores = cca_analysis(trial_data, TARGET_FREQS, sfreq)
            pred_cca = max(cca_scores, key=cca_scores.get)
            row_base['Pred_CCA'] = pred_cca
            row_base['Acc_CCA'] = int(pred_cca == info_c['Frequency'])
            row_base['Score_CCA'] = float(cca_scores[pred_cca])
            row_base['Margin_CCA'] = score_margin(cca_scores)

            fbcca_scores = fbcca_analysis(trial_data, TARGET_FREQS, sfreq)
            pred_fbcca = max(fbcca_scores, key=fbcca_scores.get)
            row_base['Pred_FBCCA'] = pred_fbcca
            row_base['Acc_FBCCA'] = int(pred_fbcca == info_c['Frequency'])
            row_base['Score_FBCCA'] = float(fbcca_scores[pred_fbcca])
            row_base['Margin_FBCCA'] = score_margin(fbcca_scores)

            train_mask = np.ones(len(data_all), dtype=bool)
            train_mask[i] = False
            filters, templates = train_trca_filters(data_all[train_mask], labels[train_mask], TARGET_FREQS)
            trca_scores = trca_classify(data_all[i], filters, templates)
            pred_trca = max(trca_scores, key=trca_scores.get)
            row_base['Pred_TRCA'] = pred_trca
            row_base['Acc_TRCA'] = int(pred_trca == info_c['Frequency'])
            row_base['Score_TRCA'] = float(trca_scores[pred_trca])
            row_base['Margin_TRCA'] = score_margin(trca_scores)

            fb_filters, fb_templates = train_fbtrca_filters(data_all[train_mask], labels[train_mask], TARGET_FREQS,
                                                            sfreq)
            fbtrca_scores = fbtrca_classify(data_all[i], fb_filters, fb_templates, TARGET_FREQS, sfreq)
            pred_fbtrca = max(fbtrca_scores, key=fbtrca_scores.get)
            row_base['Pred_FBTRCA'] = pred_fbtrca
            row_base['Acc_FBTRCA'] = int(pred_fbtrca == info_c['Frequency'])
            row_base['Score_FBTRCA'] = float(fbtrca_scores[pred_fbtrca])
            row_base['Margin_FBTRCA'] = score_margin(fbtrca_scores)

            result['classification_rows'].append(row_base)

        result['Status'] = 'ok'

    except Exception as e:
        result['Status'] = 'error'
        result['Error'] = str(e)

    return result

## 4. Batch Processing

In [ ]:
all_results = []
for subj in all_subjects:
    print(f'Processing sub-{subj} ...', end=' ')
    res = process_subject(subj)
    all_results.append(res)
    print(res['Status'])

df_status = pd.DataFrame([{
    'Subject': r['Subject'],
    'Status': r['Status'],
    'Error': r['Error'],
    'N_Trials': r['N_Trials'],
    'N_Channels': r['N_Channels'],
    'Sfreq': r['Sfreq'],
} for r in all_results])

print(f'\nSuccess: {(df_status["Status"] == "ok").sum()}, Failed: {(df_status["Status"] != "ok").sum()}')
display(df_status)

## 5. Merge Subject Data

In [ ]:
ok_results = [r for r in all_results if r['Status'] == 'ok']

df_snr = pd.DataFrame([row for r in ok_results for row in r['snr_rows']])
df_snr_harmonics = pd.DataFrame([row for r in ok_results for row in r['snr_harmonic_rows']])
df_quality = pd.DataFrame([row for r in ok_results for row in r['quality_rows']])
df_itpc = pd.DataFrame([row for r in ok_results for row in r['itpc_rows']])
df_class = pd.DataFrame([row for r in ok_results for row in r['classification_rows']])

subj_psd_list = []
for r in ok_results:
    if r.get('avg_psd') is not None:
        subj_psd_list.append({
            'Subject': r['Subject'],
            'avg_psd': r['avg_psd'],
            'psd_freqs': r['psd_freqs'],
        })

print(
    f'SNR: {len(df_snr)} rows, SNR Harmonics: {len(df_snr_harmonics)} rows, Quality: {len(df_quality)} rows, ITPC: {len(df_itpc)} rows, Classification: {len(df_class)} rows')
print(f'PSD data: {len(subj_psd_list)} subjects')
display(df_snr.head())
display(df_snr_harmonics.head())
display(df_quality.head())
display(df_itpc.head())
display(df_class.head())

## 6. Grand-Average Spectrum & SNR

All subjects' EEG signals are z-scored and grand-averaged. Solid lines = group mean; shaded areas = SEM.

In [ ]:
freqs = subj_psd_list[0]['psd_freqs']

all_zscored = []
for item in subj_psd_list:
    ch_avg = item['avg_psd'].mean(axis=0)
    mu, sigma = ch_avg.mean(), ch_avg.std()
    zscored = (ch_avg - mu) / (sigma + 1e-10)
    all_zscored.append(zscored)

all_zscored = np.array(all_zscored)
grand_mean = all_zscored.mean(axis=0)
grand_sem = all_zscored.std(axis=0) / np.sqrt(len(all_zscored))

plt.figure(figsize=(10, 5))
plt.plot(freqs, grand_mean, linewidth=2.5, color='steelblue', label='Grand Mean (z-scored)')
plt.fill_between(freqs, grand_mean - grand_sem, grand_mean + grand_sem,
                 alpha=0.3, color='steelblue', label='± SEM')
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
for tf in TARGET_FREQS:
    plt.axvline(x=tf, color='red', linestyle=':', alpha=0.5, linewidth=1)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Normalized PSD (z-score)')
plt.title(f'Grand Average Power Spectrum (N={len(all_zscored)})')
plt.legend()
plt.xlim(5, MAX_ANALYSIS_FREQ)
plt.show()

## 7. Subject-Level Data Quality Overview

In [ ]:
subj_summary = df_snr.groupby('Subject').agg(
    Trials=('SNR_dB', 'size'),
    Mean_SNR_dB=('SNR_dB', 'mean'),
    SD_SNR_dB=('SNR_dB', 'std'),
    Median_SNR_dB=('SNR_dB', 'median'),
).reset_index()

subj_qc = df_quality.groupby('Subject').agg(
    Artifact_Rate=('Likely_Artifact', 'mean'),
    Mean_PTP_uV=('Mean_PTP_uV', 'mean'),
    Mean_RMS_uV=('Mean_RMS_uV', 'mean'),
).reset_index()

subj_itpc = df_itpc[df_itpc['Harmonic_Order'] == 0].groupby('Subject').agg(
    Mean_ITPC=('ITPC', 'mean'),
).reset_index()

subj_overview = subj_summary.merge(subj_qc, on='Subject').merge(subj_itpc, on='Subject', how='left')
subj_overview = subj_overview.sort_values('Mean_SNR_dB', ascending=False).reset_index(drop=True)

print('Subject-level data quality overview (sorted by Mean SNR dB, descending)')
display(subj_overview)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.barplot(data=subj_overview, x='Subject', y='Mean_SNR_dB', ax=axes[0, 0],
            order=subj_overview['Subject'])
axes[0, 0].set_title('Mean SNR (dB) by Subject')
axes[0, 0].tick_params(axis='x', rotation=45)

sns.barplot(data=subj_overview, x='Subject', y='Artifact_Rate', ax=axes[0, 1],
            order=subj_overview['Subject'])
axes[0, 1].set_title('Artifact Rate by Subject')
axes[0, 1].tick_params(axis='x', rotation=45)

sns.barplot(data=subj_overview, x='Subject', y='Mean_ITPC', ax=axes[1, 0],
            order=subj_overview['Subject'])
axes[1, 0].set_title('Mean Harmonic ITPC by Subject')
axes[1, 0].tick_params(axis='x', rotation=45)

sns.barplot(data=subj_overview, x='Subject', y='Mean_RMS_uV', ax=axes[1, 1],
            order=subj_overview['Subject'])
axes[1, 1].set_title('Mean RMS (uV) by Subject')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 8. Group SNR Analysis

In [ ]:
condition_summary = (
    df_snr
    .groupby(['Frequency', 'Transparency', 'Color'])
    .agg(
        N_Subjects=('Subject', 'nunique'),
        Trials=('SNR_dB', 'size'),
        Mean_SNR_dB=('SNR_dB', 'mean'),
        SD_SNR_dB=('SNR_dB', 'std'),
        Median_SNR_dB=('SNR_dB', 'median'),
    )
    .reset_index()
)
print('Group condition-level SNR summary')
display(condition_summary)

plt.figure(figsize=(10, 5))
sns.barplot(data=df_snr, x='Frequency', y='SNR_dB', hue='Transparency', errorbar='se')
plt.title('SNR by Frequency and Transparency (Group)')
plt.ylabel('SNR (dB)')
plt.legend(title='Transparency')
plt.show()

plt.figure(figsize=(10, 5))
sns.barplot(data=df_snr, x='Frequency', y='SNR_dB', hue='Color', errorbar='se')
plt.title('SNR by Frequency and Color (Group)')
plt.ylabel('SNR (dB)')
plt.legend(title='Color')
plt.show()

## 9. Subject x Condition SNR Heatmap

In [ ]:
snr_pivot = (
    df_snr
    .groupby(['Subject', 'Frequency', 'Transparency', 'Color'])['SNR_dB']
    .mean()
    .reset_index()
)
snr_pivot['Condition'] = (
        snr_pivot['Frequency'].astype(int).astype(str) + 'Hz_' +
        'trans' + snr_pivot['Transparency'].astype(str) + '_' +
        snr_pivot['Color']
)

heatmap_data = snr_pivot.pivot_table(index='Subject', columns='Condition', values='SNR_dB')
heatmap_data = heatmap_data.reindex(
    heatmap_data.mean(axis=1).sort_values(ascending=False).index
)

plt.figure(figsize=(16, max(6, len(heatmap_data) * 0.5)))
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            linewidths=0.5, cbar_kws={'label': 'SNR (dB)'})
plt.title('SNR (dB) by Subject and Condition')
plt.xlabel('Condition')
plt.ylabel('Subject')
plt.tight_layout()
plt.show()

## 10. Group Trial Quality Control

In [ ]:
qc_condition = (
    df_quality
    .groupby(['Frequency', 'Transparency', 'Color'])
    .agg(
        N_Subjects=('Subject', 'nunique'),
        Trials=('Trial_Index', 'size'),
        Artifact_Rate=('Likely_Artifact', 'mean'),
        Mean_PTP_uV=('Mean_PTP_uV', 'mean'),
        Max_PTP_P95=('Max_PTP_uV', lambda x: np.nanpercentile(x, 95)),
        Mean_RMS_uV=('Mean_RMS_uV', 'mean'),
        Max_RMS_P95=('Max_RMS_uV', lambda x: np.nanpercentile(x, 95)),
    )
    .reset_index()
)
print('Group condition-level QC summary')
display(qc_condition)

qc_subj = (
    df_quality
    .groupby('Subject')
    .agg(
        Trials=('Trial_Index', 'size'),
        Artifact_Rate=('Likely_Artifact', 'mean'),
        Mean_PTP_uV=('Mean_PTP_uV', 'mean'),
        Mean_RMS_uV=('Mean_RMS_uV', 'mean'),
    )
    .reset_index()
    .sort_values('Artifact_Rate', ascending=False)
)
print('\nSubject-level QC summary (sorted by Artifact Rate, descending)')
display(qc_subj)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df_quality, x='Subject', y='Max_PTP_uV', ax=axes[0])
axes[0].axhline(QC_PTP_UV_THRESHOLD, color='red', linestyle='--', alpha=0.6)
axes[0].set_title('Max PTP by Subject')
axes[0].set_ylabel('Max PTP (uV)')
axes[0].tick_params(axis='x', rotation=45)

sns.boxplot(data=df_quality, x='Subject', y='Max_RMS_uV', ax=axes[1])
axes[1].axhline(QC_RMS_UV_THRESHOLD, color='red', linestyle='--', alpha=0.6)
axes[1].set_title('Max RMS by Subject')
axes[1].set_ylabel('Max RMS (uV)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 11. Group ITPC Analysis

In [ ]:
df_itpc_harmonic = df_itpc[df_itpc['Harmonic_Order'] > 0]

itpc_condition = (
    df_itpc[df_itpc['Harmonic_Order'] == 0]
    .groupby(['Frequency', 'Transparency', 'Color'])
    .agg(
        N_Subjects=('Subject', 'nunique'),
        Mean_ITPC=('ITPC', 'mean'),
        SD_ITPC=('ITPC', 'std'),
    )
    .reset_index()
)
print('Group condition-level Mean Harmonic ITPC')
display(itpc_condition)

plt.figure(figsize=(10, 5))
sns.barplot(data=df_itpc_harmonic, x='Frequency', y='ITPC', hue='Harmonic_Order',
            errorbar='se')
plt.title('ITPC by Frequency and Harmonic (Group)')
plt.ylim(0, 1.05)
plt.legend(title='Harmonic Order')
plt.show()

plt.figure(figsize=(10, 5))
sns.barplot(
    data=df_itpc[df_itpc['Harmonic_Order'] == 0],
    x='Transparency', y='ITPC', hue='Color',
    errorbar='se'
)
plt.title('Mean Harmonic ITPC by Transparency and Color (Group)')
plt.ylim(0, 1.05)
plt.legend(title='Color')
plt.show()

subj_itpc_mean = (
    df_itpc[df_itpc['Harmonic_Order'] == 0]
    .groupby(['Subject', 'Frequency'])['ITPC']
    .mean()
    .reset_index()
)
plt.figure(figsize=(10, 6))
sns.boxplot(data=subj_itpc_mean, x='Frequency', y='ITPC')
sns.stripplot(data=subj_itpc_mean, x='Frequency', y='ITPC',
              color='black', alpha=0.5, size=4)
plt.title('Group ITPC Distribution by Frequency')
plt.ylim(0, 1.05)
plt.show()

## 12. Group Multi-Factor ANOVA

Using within-subject condition means as observations, test main and interaction effects of frequency, transparency, and color on SNR and ITPC.

In [ ]:
subj_cond_snr = (
    df_snr
    .groupby(['Subject', 'Frequency', 'Transparency', 'Color'])['SNR_dB']
    .mean()
    .reset_index()
)

subj_cond_snr['Frequency'] = subj_cond_snr['Frequency'].astype(str)
subj_cond_snr['Transparency'] = subj_cond_snr['Transparency'].astype(str)

model_snr = smf.ols(
    'SNR_dB ~ C(Frequency) * C(Transparency) * C(Color)',
    data=subj_cond_snr,
).fit()
anova_snr = sm.stats.anova_lm(model_snr, typ=2)

print('SNR multi-factor ANOVA (group)')
display(anova_snr)

subj_cond_itpc = (
    df_itpc[df_itpc['Harmonic_Order'] == 0]
    .groupby(['Subject', 'Frequency', 'Transparency', 'Color'])['ITPC']
    .mean()
    .reset_index()
)

subj_cond_itpc['Frequency'] = subj_cond_itpc['Frequency'].astype(str)
subj_cond_itpc['Transparency'] = subj_cond_itpc['Transparency'].astype(str)

model_itpc = smf.ols(
    'ITPC ~ C(Frequency) * C(Transparency) * C(Color)',
    data=subj_cond_itpc,
).fit()
anova_itpc = sm.stats.anova_lm(model_itpc, typ=2)

print('\nITPC multi-factor ANOVA (group)')
display(anova_itpc)

## 13. Outlier Subject Identification

Identify subjects with significantly low SNR or high artifact rates based on group statistics.

In [ ]:
group_snr_mean = subj_overview['Mean_SNR_dB'].mean()
group_snr_std = subj_overview['Mean_SNR_dB'].std()
snr_low_threshold = group_snr_mean - 1.5 * group_snr_std

group_artifact_mean = subj_overview['Artifact_Rate'].mean()
group_artifact_std = subj_overview['Artifact_Rate'].std()
artifact_high_threshold = group_artifact_mean + 1.5 * group_artifact_std

outlier_snr = subj_overview[subj_overview['Mean_SNR_dB'] < snr_low_threshold]
outlier_artifact = subj_overview[subj_overview['Artifact_Rate'] > artifact_high_threshold]

print(f'Group Mean SNR: {group_snr_mean:.2f} +/- {group_snr_std:.2f} dB')
print(f'SNR low outlier threshold (< mean - 1.5*SD): {snr_low_threshold:.2f} dB')
print(f'Group Artifact Rate: {group_artifact_mean:.3f} +/- {group_artifact_std:.3f}')
print(f'Artifact high outlier threshold (> mean + 1.5*SD): {artifact_high_threshold:.3f}')

if not outlier_snr.empty:
    print(f'\nSubjects with significantly low SNR ({len(outlier_snr)}):')
    display(outlier_snr[['Subject', 'Mean_SNR_dB', 'Trials']])
else:
    print('\nNo subjects with significantly low SNR found')

if not outlier_artifact.empty:
    print(f'\nSubjects with significantly high artifact rate ({len(outlier_artifact)}):')
    display(outlier_artifact[['Subject', 'Artifact_Rate', 'Trials']])
else:
    print('\n未发现Subjects with significantly high artifact rate')

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(subj_overview['Mean_SNR_dB'], subj_overview['Artifact_Rate'], s=80)
for _, row in subj_overview.iterrows():
    ax.annotate(row['Subject'], (row['Mean_SNR_dB'], row['Artifact_Rate']),
                textcoords='offset points', xytext=(5, 5), fontsize=9)
ax.axvline(snr_low_threshold, color='red', linestyle='--', alpha=0.6, label='SNR threshold')
ax.axhline(artifact_high_threshold, color='orange', linestyle='--', alpha=0.6, label='Artifact threshold')
ax.set_xlabel('Mean SNR (dB)')
ax.set_ylabel('Artifact Rate')
ax.set_title('Subject Quality: SNR vs Artifact Rate')
ax.legend()
plt.tight_layout()
plt.show()

## 14. SNR-ITPC Correlation

In [ ]:
subj_cond_both = (
    df_snr
    .groupby(['Subject', 'Frequency', 'Transparency', 'Color'])['SNR_dB']
    .mean()
    .reset_index()
    .merge(
        df_itpc[df_itpc['Harmonic_Order'] == 0]
        .groupby(['Subject', 'Frequency', 'Transparency', 'Color'])['ITPC']
        .mean()
        .reset_index(),
        on=['Subject', 'Frequency', 'Transparency', 'Color'],
    )
)

corr_val = subj_cond_both['SNR_dB'].corr(subj_cond_both['ITPC'])
print(f'Pearson correlation between SNR_dB and ITPC: {corr_val:.3f}')

plt.figure(figsize=(8, 6))
sns.scatterplot(data=subj_cond_both, x='SNR_dB', y='ITPC', hue='Frequency', palette='deep')
plt.title(f'SNR vs ITPC (r = {corr_val:.3f})')
plt.xlabel('Mean SNR (dB)')
plt.ylabel('Mean Harmonic ITPC')
plt.tight_layout()
plt.show()

## 15. Group Classification Performance Overview

Per-subject frequency recognition using CCA, FBCCA, TRCA, and FBTRCA (Leave-One-Out); summary of group-level accuracy and ITR.

In [ ]:
algorithms = ['CCA', 'FBCCA', 'TRCA', 'FBTRCA']

algo_summary_rows = []
for algo in algorithms:
    acc_col = f'Acc_{algo}'
    margin_col = f'Margin_{algo}'
    if acc_col not in df_class.columns:
        continue
    acc = df_class[acc_col].mean()
    margin = df_class[margin_col].mean()
    algo_summary_rows.append({
        'Algorithm': algo,
        'Accuracy': acc,
        'ITR_WindowOnly_bits_min': calculate_itr(len(TARGET_FREQS), acc, DECISION_WINDOW_S),
        'ITR_FullTrialCycle_bits_min': calculate_itr(len(TARGET_FREQS), acc, FULL_TRIAL_CYCLE_S),
        'Mean_Score_Margin': margin,
    })

df_algo_summary = pd.DataFrame(algo_summary_rows)
print('=== Group algorithm performance overview ===')
display(df_algo_summary)

present_algos = [a for a in algorithms if f'Acc_{a}' in df_class.columns]
for algo in present_algos:
    print(f'\n=== {algo}: Group condition-level accuracy ===')
    display(summarize_accuracy(df_class, ['Frequency', 'Transparency', 'Color'], acc_col=f'Acc_{algo}'))

n_algo = len(present_algos)
fig, axes = plt.subplots(1, n_algo, figsize=(5 * n_algo, 5))
if n_algo == 1:
    axes = [axes]
for idx, algo in enumerate(present_algos):
    sns.barplot(data=df_class, x='Frequency', y=f'Acc_{algo}', hue='Transparency',
                errorbar='se', ax=axes[idx])
    axes[idx].set_title(f'{algo} Accuracy by Frequency x Transparency')
    axes[idx].set_ylabel('Accuracy')
    axes[idx].set_ylim(0, 1.05)
    axes[idx].legend(title='Transparency')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, n_algo, figsize=(5 * n_algo, 5))
if n_algo == 1:
    axes = [axes]
for idx, algo in enumerate(present_algos):
    sns.barplot(data=df_class, x='Frequency', y=f'Acc_{algo}', hue='Color',
                errorbar='se', ax=axes[idx])
    axes[idx].set_title(f'{algo} Accuracy by Frequency x Color')
    axes[idx].set_ylabel('Accuracy')
    axes[idx].set_ylim(0, 1.05)
    axes[idx].legend(title='Color')

plt.tight_layout()
plt.show()

## 16. Subject-Level Classification Accuracy

In [ ]:
subj_class = df_class.groupby('Subject').agg(
    Trials=('Trial_Index', 'size'),
    **{f'Acc_{a}': (f'Acc_{a}', 'mean') for a in algorithms if f'Acc_{a}' in df_class.columns},
    **{f'Margin_{a}': (f'Margin_{a}', 'mean') for a in algorithms if f'Margin_{a}' in df_class.columns},
).reset_index()

present_algos = [a for a in algorithms if f'Acc_{a}' in df_class.columns]
for algo in present_algos:
    subj_class[f'ITR_{algo}_bits_min'] = subj_class[f'Acc_{algo}'].apply(
        lambda a: calculate_itr(len(TARGET_FREQS), a, DECISION_WINDOW_S)
    )

best_algo_name = df_algo_summary.loc[df_algo_summary['Accuracy'].idxmax(), 'Algorithm']
subj_class = subj_class.sort_values(f'Acc_{best_algo_name}', ascending=False).reset_index(drop=True)
print(f'Subject-level classification accuracy (sorted by {best_algo_name} accuracy, descending)')
display(subj_class)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

acc_cols = [f'Acc_{a}' for a in present_algos]
melted_acc = subj_class.melt(
    id_vars='Subject', value_vars=acc_cols,
    var_name='Algorithm', value_name='Accuracy',
)
melted_acc['Algorithm'] = melted_acc['Algorithm'].str.replace('Acc_', '')

sns.barplot(data=melted_acc, x='Subject', y='Accuracy', hue='Algorithm', ax=axes[0])
axes[0].set_title('Classification Accuracy by Subject and Algorithm')
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis='x', rotation=45)

itr_cols = [f'ITR_{a}_bits_min' for a in present_algos]
melted_itr = subj_class.melt(
    id_vars='Subject', value_vars=itr_cols,
    var_name='Algorithm', value_name='ITR_bits_min',
)
melted_itr['Algorithm'] = melted_itr['Algorithm'].str.replace('ITR_', '').str.replace('_bits_min', '')

sns.barplot(data=melted_itr, x='Subject', y='ITR_bits_min', hue='Algorithm', ax=axes[1])
axes[1].set_title('ITR (bits/min) by Subject and Algorithm')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 17. Group Confusion Matrix

In [ ]:
present_algos = [a for a in algorithms if f'Pred_{a}' in df_class.columns]
n_algo = len(present_algos)

fig, axes = plt.subplots(1, n_algo, figsize=(5 * n_algo, 5))
if n_algo == 1:
    axes = [axes]
for idx, algo in enumerate(present_algos):
    cm = pd.crosstab(df_class['True_Freq'], df_class[f'Pred_{algo}'],
                     rownames=['True'], colnames=['Pred'], normalize='index')
    sns.heatmap(cm, annot=True, fmt='.1%', cmap='Blues',
                xticklabels=TARGET_FREQS, yticklabels=TARGET_FREQS, ax=axes[idx])
    axes[idx].set_title(f'{algo} Confusion Matrix (Group)')
    axes[idx].set_xlabel('Pred')
    axes[idx].set_ylabel('True')

plt.tight_layout()
plt.show()

## 18. Classification Accuracy Factor Analysis

In [ ]:
present_algos = [a for a in algorithms if f'Acc_{a}' in df_class.columns]
for algo in present_algos:
    print(f'\n=== {algo} accuracy ANOVA ===')
    subj_cond_acc = (
        df_class
        .groupby(['Subject', 'Frequency', 'Transparency', 'Color'])[f'Acc_{algo}']
        .mean()
        .reset_index()
    )
    subj_cond_acc['Frequency'] = subj_cond_acc['Frequency'].astype(str)
    subj_cond_acc['Transparency'] = subj_cond_acc['Transparency'].astype(str)

    model = smf.ols(
        f'Acc_{algo} ~ C(Frequency) * C(Transparency) * C(Color)',
        data=subj_cond_acc,
    ).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    display(anova_table)

## 19. Classification Accuracy vs SNR

In [ ]:
present_algos = [a for a in algorithms if f'Acc_{a}' in df_class.columns]
acc_cols_for_merge = [f'Acc_{a}' for a in present_algos]

subj_snr_acc = (
    df_snr
    .groupby(['Subject', 'Frequency'])['SNR_dB']
    .mean()
    .reset_index()
    .merge(
        df_class.groupby(['Subject', 'Frequency'])[acc_cols_for_merge].mean().reset_index(),
        on=['Subject', 'Frequency'],
    )
)

n_algo = len(present_algos)
fig, axes = plt.subplots(1, n_algo, figsize=(5 * n_algo, 5))
if n_algo == 1:
    axes = [axes]
for idx, algo in enumerate(present_algos):
    corr_val = subj_snr_acc['SNR_dB'].corr(subj_snr_acc[f'Acc_{algo}'])
    sns.scatterplot(data=subj_snr_acc, x='SNR_dB', y=f'Acc_{algo}',
                    hue='Frequency', palette='deep', ax=axes[idx])
    axes[idx].set_title(f'{algo}: SNR vs Accuracy (r={corr_val:.3f})')
    axes[idx].set_xlabel('Mean SNR (dB)')
    axes[idx].set_ylabel('Accuracy')
    axes[idx].legend(title='Frequency')
plt.tight_layout()
plt.show()

print('Pearson correlation between SNR and classification accuracy:')
for algo in present_algos:
    r = subj_snr_acc['SNR_dB'].corr(subj_snr_acc[f'Acc_{algo}'])
    print(f'  {algo}: r = {r:.3f}')

## 20. Save Results

In [ ]:
output_dir = os.path.join(os.path.dirname(BIDS_ROOT), 'group_analysis_results')
os.makedirs(output_dir, exist_ok=True)

subj_overview.to_csv(os.path.join(output_dir, 'subject_overview.csv'), index=False)
condition_summary.to_csv(os.path.join(output_dir, 'condition_snr_summary.csv'), index=False)
qc_condition.to_csv(os.path.join(output_dir, 'condition_qc_summary.csv'), index=False)
itpc_condition.to_csv(os.path.join(output_dir, 'condition_itpc_summary.csv'), index=False)
df_snr.to_csv(os.path.join(output_dir, 'trial_level_snr.csv'), index=False)
df_quality.to_csv(os.path.join(output_dir, 'trial_level_quality.csv'), index=False)
df_itpc.to_csv(os.path.join(output_dir, 'condition_level_itpc.csv'), index=False)
df_class.to_csv(os.path.join(output_dir, 'trial_level_classification.csv'), index=False)
df_algo_summary.to_csv(os.path.join(output_dir, 'algorithm_summary.csv'), index=False)
subj_class.to_csv(os.path.join(output_dir, 'subject_classification.csv'), index=False)

print(f'Results saved to: {output_dir}')